# IEE 575 — GP Review
**Tips:**
1. Try writing the code yourself before looking anything up.
2. If you are stuck on syntax, check the sklearn docs.
3. Think about *why* the output looks the way it does — not just whether the code runs.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, ConstantKernel as C, DotProduct, ExpSineSquared, WhiteKernel
)

np.random.seed(575)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Setup complete.')

---
## Part 1 — The Function and the Data

We will work with this function throughout:

$$f(x) = \sin(2x) + 0.3\cos(5x), \quad x \in [-3, 3]$$

We observe it at 6 locations with a small amount of noise.


In [ ]:
def f(x):
    return np.sin(2*x) + 0.3*np.cos(5*x)

X_train = np.array([[-2.5], [-1.5], [-0.5], [0.8], [1.8], [2.7]])
y_train = f(X_train.flatten()) + np.random.randn(6) * 0.15

x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

plt.figure(figsize=(10, 3))
plt.plot(x_plot, f(x_plot), 'k--', label='True f(x)')
plt.scatter(X_train, y_train, c='red', zorder=5, label='Observations')
plt.legend()
plt.xlabel('x')
plt.title('The function and our observations')
plt.tight_layout()
plt.show()

---
## Part 2 — Fit a GP with an RBF Kernel

Fit a GP to the training data using a **RBF kernel**.

Use `normalize_y=True` and `n_restarts_optimizer=5`.

Produce a plot showing:
- the posterior mean
- ±2σ uncertainty band
- training observations
- true function

Print the kernel hyperparameters **before and after** fitting.


In [ ]:
# Define kernel
kernel = ...
print('Before:', kernel)

# Fit
gp_rbf = ...
print('After: ', gp_rbf.kernel_)

# Predict
mu, std = ...

# Plot


**Q2.1** What did sklearn change in the hyperparameters after fitting? What does that tell you about the data?

> *Your answer here.*


**Q2.2** Look at the uncertainty band. Where is it widest? Where is it narrowest? Explain why.

> *Your answer here.*


---
## Part 3 — Why Does the RBF Struggle?

Now deliberately break it. Fit the same RBF kernel but **disable the optimizer** so the length scale stays at its default value of 1.0.

```python
gp = GaussianProcessRegressor(kernel=RBF(length_scale=1.0), optimizer=None, normalize_y=True)
```

Then try the other extreme: set `length_scale=0.05` with `optimizer=None`.

Plot all three side by side: optimized, large ℓ, small ℓ.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

configs = [
    ('Optimized', GaussianProcessRegressor(kernel=C(1.0)*RBF(), normalize_y=True, n_restarts_optimizer=5)),
    ('Fixed ℓ=1.0', GaussianProcessRegressor(kernel=RBF(length_scale=1.0), optimizer=None, normalize_y=True)),
    ('Fixed ℓ=0.05', GaussianProcessRegressor(kernel=RBF(length_scale=0.05), optimizer=None, normalize_y=True)),
]

for ax, (title, gp) in zip(axes, configs):
    gp.fit(X_train, y_train)
    mu, std = gp.predict(x_plot, return_std=True)
    ax.plot(x_plot, f(x_plot), 'k--', label='True f(x)')
    ax.plot(x_plot, mu, 'b', label='Mean')
    ax.fill_between(x_plot.flatten(), mu-2*std, mu+2*std, alpha=0.2)
    ax.scatter(X_train, y_train, c='red', zorder=5)
    ax.set_title(title)
    ax.set_xlabel('x')

plt.tight_layout()
plt.show()

**Q3.1** Describe what goes wrong with the large and small length scales. Be specific about what the posterior mean and uncertainty band look like in each case.

> *Your answer here.*


**Q3.2** The std at training points is small but not exactly zero. Why?

> *Your answer here.*


---
## Part 4 — Kernel Comparison

Fit three separate GPs on the same training data:

1. Matérn ν=3/2
2. RBF
3. Linear (`DotProduct`)

Plot all three. Print the **log marginal likelihood** for each.

```python
# After fitting:
gp.log_marginal_likelihood_value_
```


In [ ]:
kernels = {
    'Matern 3/2': C(1.0) * Matern(nu=1.5),
    'RBF':        C(1.0) * RBF(),
    'Linear':     C(1.0) * DotProduct(),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (name, kernel) in zip(axes, kernels.items()):
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
    gp.fit(X_train, y_train)
    mu, std = gp.predict(x_plot, return_std=True)
    ax.plot(x_plot, f(x_plot), 'k--')
    ax.plot(x_plot, mu, 'b')
    ax.fill_between(x_plot.flatten(), mu-2*std, mu+2*std, alpha=0.2)
    ax.scatter(X_train, y_train, c='red', zorder=5)
    ax.set_title(f'{name}\nlog ML = {gp.log_marginal_likelihood_value_:.2f}')
    ax.set_xlabel('x')

plt.tight_layout()
plt.show()

**Q4.1** Describe what the linear kernel predicts between and beyond the training points. What prior over functions does it encode?

> *Your answer here.*


**Q4.2** Which kernel wins on log marginal likelihood? In one sentence — what does that criterion balance?

> *Your answer here.*


---
## Part 5 — Bayesian Optimization

Now we use a GP as a surrogate to **optimize** a black-box function.

The objective is:

$$g(x) = -\left(\sin(3x) + x^2 - 0.5x\right), \quad x \in [-3, 3]$$

Start with 3 observations at $x \in \{-2.5, 0.0, 2.5\}$. Run **12 iterations** of BO using Expected Improvement.

$$\text{EI}(x) = (\mu(x) - f^* - \xi)\,\Phi(Z) + \sigma(x)\,\phi(Z), \qquad Z = \frac{\mu(x) - f^* - \xi}{\sigma(x)}$$

Produce:
1. A plot of the final GP surrogate (mean, ±2σ, all observations, true function)
2. A convergence plot (best value found vs. iteration)
3. Print the query sequence


In [ ]:
def g(x):
    return -(np.sin(3*x) + x**2 - 0.5*x)

def expected_improvement(X_cand, y_obs, gp, xi=0.01):
    mu, sigma = gp.predict(X_cand, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    f_best = np.max(y_obs)
    Z = (mu - f_best - xi) / sigma
    return np.maximum((mu - f_best - xi)*norm.cdf(Z) + sigma*norm.pdf(Z), 0)

# Initial observations
X_obs = np.array([[-2.5], [0.0], [2.5]])
y_obs = g(X_obs.flatten())

x_cand = np.linspace(-3, 3, 500).reshape(-1, 1)
best_so_far = [np.max(y_obs)]
query_sequence = []

# BO loop
for i in range(12):
    # TODO: fit GP, compute EI, query next point, update X_obs and y_obs
    pass

# Plot 1: Final surrogate

# Plot 2: Convergence

# Print query sequence
print('Query sequence:', [f'{x:.3f}' for x in query_sequence])

**Q5.1** At each iteration the GP is refitted with the new observation. Name the mathematical operation this corresponds to. What two quantities does it update?

> *Your answer here.*


**Q5.2** The EI formula has two additive terms. What does each one encourage? Which drives exploitation, which drives exploration?

> *Your answer here.*


**Q5.3** Look at your query sequence. Early queries likely explore broadly; later ones cluster near the optimum. What happens to σ(x) in already-queried regions as iterations progress, and how does this shift the balance between the two EI terms?

> *Your answer here.*


**Q5.4** The convergence plot is monotonically non-decreasing. Explain in one sentence why this must always be true.

> *Your answer here.*
